In [3]:
# !pip install -U langchain-community langchain-text-splitters langchain-openai langchain-chroma

In [4]:
import os
# import urllib.request
# from langchain.document_loaders import PyPDFLoader
# from langchain.text_splitter import RecursiveCharacterTextSplitter
# from langchain.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.vectorstores import FAISS

import os
import urllib.request

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
# from langchain_chroma import Chroma

import os
from dotenv import load_dotenv

# .env파일의 환경변수를 불러옵니다.
load_dotenv()

# 환경 변수에서 API KEY를 가져옵니다.
api_key = os.getenv("OPENAI_API_KEY")

os.environ['OPENAI_API_KEY'] = api_key

C:\Users\PC\AppData\Local\Temp\ipykernel_10564\1230380972.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [5]:
urllib.request.urlretrieve("https://github.com/chatgpt-kr/openai-api-tutorial/raw/main/ch06/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf", filename="2023_북한인권보고서.pdf")

('2023_북한인권보고서.pdf', <http.client.HTTPMessage at 0x245d5d45220>)

In [6]:
loader = PyPDFLoader('2023_북한인권보고서.pdf')
pages = loader.load_and_split()
print('청크의 수:', len(pages))

청크의 수: 445


In [7]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

In [8]:
splited_docs = text_splitter.split_documents(pages)
print('분할된 청크의 수:', len(splited_docs))

분할된 청크의 수: 497


In [9]:
chunks = [splited_doc.page_content for splited_doc in splited_docs]
print('청크의 최대 길이 :',max(len(chunk) for chunk in chunks))

청크의 최대 길이 : 1000


In [10]:
db = Chroma.from_documents(splited_docs, OpenAIEmbeddings())
print('문서의 수:', db._collection.count())

문서의 수: 497


In [11]:
question = '북한의 교육과정'
docs = db.similarity_search(question)
print('문서의 수:', len(docs))

문서의 수: 4


In [12]:
# print(docs)
for doc in docs:
  print(doc.page_content[:100])
  print('--' * 10)

2023 북한인권보고서
40
명목의 교육비용이 전가되고 있는 것으로 나타났다. 교과서는 ‘교과
서 요금’이라는 명목으로 일정 금액을 내야하는 경우가 많으며, 교
과서가 모든 학생에
--------------------
309		북한의	학제는	2012년	전반적	의무교육(유치원	1년,	소학교	5년,	초급중학교	3년,	고급중학교	3년)으로	
개편되었는데,	학제개편	이전에는	초급중학교와	고급중학교를	통
--------------------
2023 북한인권보고서
342
2018년에 학교에서 추천하여 소년궁전 스키부에 선발되었으나, 체
육종합지도원이 자신의 출신성분이 좋지 않다는 이유로 선발명단에
서 자신을 제외했다고
--------------------
2023 북한인권보고서
184
데, 당국이 실시하는 반종교 교육을 통해 기독교를 접한 경우였다. 기
독교 관련 북한당국의 반종교 교육은 학교 교과과정에서 뿐만 아니
라 졸업 후 조
--------------------


In [ ]:
# ChromaDB 초기화
# ChromaDB 객체를 사용 중이라면 참조 제거
# del db

# import gc
# gc.collect()

In [ ]:

# from pathlib import Path
# import shutil
# import chromadb

# persist_dir = Path("./chroma_test.db")

# if persist_dir.exists():
#     shutil.rmtree(persist_dir)

# client = chromadb.PersistentClient(path=str(persist_dir))

In [15]:
db_to_file = Chroma.from_documents(splited_docs, OpenAIEmbeddings(), persist_directory = './chroma_test.db')
print('문서의 수:', db_to_file._collection.count())

문서의 수: 497


In [16]:
db_from_file = Chroma(persist_directory='./chroma_test.db',
		      embedding_function=OpenAIEmbeddings())

C:\Users\PC\AppData\Local\Temp\ipykernel_10564\112611589.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db_from_file = Chroma(persist_directory='./chroma_test.db',


In [17]:
print('문서의 수:', db_from_file._collection.count())

문서의 수: 497


In [18]:
faiss_db = FAISS.from_documents(splited_docs, OpenAIEmbeddings())
print('문서의 수:', faiss_db.index.ntotal)

문서의 수: 497


In [21]:
faiss_db.save_local('faiss_index')

new_db_faiss = FAISS.load_local('faiss_index',
				OpenAIEmbeddings(),
				allow_dangerous_deserialization=True)

In [23]:
question = '북한의 교육 과정'
docs = new_db_faiss.similarity_search(question)

for doc in docs:
  print(doc.page_content[:200])
  print('--' * 10)

2023 북한인권보고서
40
명목의 교육비용이 전가되고 있는 것으로 나타났다. 교과서는 ‘교과
서 요금’이라는 명목으로 일정 금액을 내야하는 경우가 많으며, 교
과서가 모든 학생에게 충분히 제공되지 않고 학년을 마치면 다음 학
년에 교과서를 물려주어야 했다는 사례가 다수 수집되었다. 소학교
부터 학교운영비, 꼬마계획 등의 비용을 내야했다는 진술이 꾸준히 

--------------------
309		북한의	학제는	2012년	전반적	의무교육(유치원	1년,	소학교	5년,	초급중학교	3년,	고급중학교	3년)으로	
개편되었는데,	학제개편	이전에는	초급중학교와	고급중학교를	통합하여	중학교	6년	과정(1972년~2011
년)으로	운영하였고,	중학교	또는	고등중학교라고	칭하였다.(통일부	국립통일교육원,	『북한의	이해』,	
2022)
--------------------
2023 북한인권보고서
184
데, 당국이 실시하는 반종교 교육을 통해 기독교를 접한 경우였다. 기
독교 관련 북한당국의 반종교 교육은 학교 교과과정에서 뿐만 아니
라 졸업 후 조직생활을 통해서도 이루어지고 있었다. 수집된 증언에 
따르면 북한에서 반종교 교육을 받고 종교에 대한 부정적 인식이 증
가했다고 한다. 기독교를 믿는 사람을 반동분자로 인식하고 있
--------------------
2023 북한인권보고서
342
2018년에 학교에서 추천하여 소년궁전 스키부에 선발되었으나, 체
육종합지도원이 자신의 출신성분이 좋지 않다는 이유로 선발명단에
서 자신을 제외했다고 진술하였다. 정치범수용소에서는 이주민 자
녀의 경우 정규교육과정을 받지 못한다는 증언도 있었다. 정치범수
용소에도 소학교와 중학교가 있지만 일반 학교와는 달리 학생들이 
책가방 
--------------------
